In [ ]:
import equinox as eqx
import jax.numpy as jnp
from context_flux_no.nn.embedding import PatchEmbedding
from jaxtyping import Array, Float

In [ ]:
def get_grid_1d(x: Float[Array, "... dim"]) -> Float[Array, "... 1"]:
    *rest, n_x, _ = x.shape
    grid_x = jnp.expand_dims(jnp.linspace(0, 1, n_x), axis=-1)
    return jnp.broadcast_to(grid_x, shape=(*rest, n_x, 1))


class AbstractPositionEncoding(eqx.Module):
    pass


class LearnedPositionEncoding(AbstractPositionEncoding):
    encodings: Float[Array, "*shapes embedding_dim"]

    def __init__(self, position_shape: tuple[int, ...], encoding_dim: int):
        self.encodings = jnp.zeros(
            (*position_shape, encoding_dim),
        )

    def __call__(self, x: Float[Array, "*shapes embedding_dim"]):
        # TODO: need to assert the input shape matches the position_embedding
        return x + self.encodings


class DPOT1D(eqx.Module):
    patch_embedding: PatchEmbedding
    position_embedding: AbstractPositionEncoding
    time_aggregator: TimeAggregator
    blocks: list[DPOTBlock]
    output_head: eqx.nn.Sequential
    cls_head: eqx.nn.Sequential

    def __call__(
        self, u: Float[Array, "time channels grid_x"]
    ) -> Float[Array, "channels grid_x"]:
        # TODO: Check if normalization is actually used and implement if necessary
        grid = get_grid_1d(u)
        u: Float[Array, "time grid_x dim+1"] = jnp.concatenate((u, grid), axis=-1)

        # Patch embedding for the spatial dimensions
        v = eqx.filter_vmap(self.patch_embedding)(u)

        # Apply positional embedding
        v = eqx.filter_vmap(self.position_embedding)(v)

        # Time aggregation layer
        v: Float[Array, "latent_dim patch_x"] = self.time_aggregator(v)

        for dpot_block in self.blocks:
            v = dpot_block(v)

        u_next = self.output_head(v)

        cls_token = jnp.mean(v, axis=-1)
        cls_pred = self.cls_head(cls_token)
        return u_next, cls_pred